# 07 — Logistics Agent

Tests `agents/logistics_agent.py`:
- `_lookup()` for all 25 Sri Lankan districts
- Tier classification (Tier 1 = next-day, Tier 2 = 2–3 days, Tier 3 = 3–5 days)
- Not-covered districts return the right info
- `run()` for known locations, unknown locations (LLM fallback), and deadline handling
- Case/whitespace insensitivity

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

In [ ]:
from agents.logistics_agent import _lookup, run, DISTRICT_COVERAGE, TIER_DESCRIPTION

## 1. DISTRICT_COVERAGE data structure

In [ ]:
print(f'Districts defined: {len(DISTRICT_COVERAGE)}')
print('Sample entries:')
for district, info in list(DISTRICT_COVERAGE.items())[:5]:
    print(f'  {district}: {info}')

assert len(DISTRICT_COVERAGE) == 25, f'Expected 25 Sri Lankan districts, got {len(DISTRICT_COVERAGE)}'

for district, info in DISTRICT_COVERAGE.items():
    assert 'tier' in info, f'{district} missing tier'
    assert 'covered' in info, f'{district} missing covered'

print('\nTier descriptions:')
for tier, desc in TIER_DESCRIPTION.items():
    print(f'  Tier {tier}: {desc}')

print('District structure: PASSED')

## 2. Tier 1 districts (next-day delivery)

In [ ]:
tier1_districts = ['Colombo', 'Gampaha', 'Kalutara']

for district in tier1_districts:
    info = _lookup(district)
    print(f'{district}: {info}')
    assert info is not None, f'{district} should be found'
    assert info['covered'] is True, f'{district} should be covered'
    assert info['tier'] == 1, f'{district} should be Tier 1'

print('Tier 1 districts: PASSED')

## 3. Tier 2 districts (2–3 day delivery)

In [ ]:
tier2_samples = ['Kandy', 'Galle', 'Kurunegala']

for district in tier2_samples:
    info = _lookup(district)
    print(f'{district}: {info}')
    assert info is not None
    assert info['covered'] is True
    assert info['tier'] == 2, f'{district} should be Tier 2'

print('Tier 2 districts: PASSED')

## 4. Tier 3 districts (3–5 day delivery)

In [ ]:
tier3_samples = ['Jaffna', 'Batticaloa', 'Trincomalee']

for district in tier3_samples:
    info = _lookup(district)
    print(f'{district}: {info}')
    assert info is not None
    assert info['covered'] is True
    assert info['tier'] == 3, f'{district} should be Tier 3'

print('Tier 3 districts: PASSED')

## 5. Not-covered districts

In [ ]:
not_covered = ['Kilinochchi', 'Mannar', 'Mullaitivu']

for district in not_covered:
    info = _lookup(district)
    print(f'{district}: {info}')
    assert info is not None, f'{district} should still be in DISTRICT_COVERAGE'
    assert info['covered'] is False, f'{district} should not be covered'

print('Not-covered districts: PASSED')

## 6. Case & whitespace insensitivity

In [ ]:
variants = ['colombo', 'COLOMBO', '  Colombo  ', 'colombO']

for variant in variants:
    info = _lookup(variant)
    print(f'_lookup({repr(variant)}) → {info}')
    assert info is not None, f'Should match regardless of case/whitespace: {repr(variant)}'
    assert info['tier'] == 1

print('Case/whitespace insensitivity: PASSED')

## 7. Unknown location returns None

In [ ]:
result = _lookup('Paris')
print(f'_lookup("Paris") → {result}')
assert result is None, 'Unknown locations should return None'

result = _lookup('')
print(f'_lookup("") → {result}')
# Empty string — should either return None or not crash
print('Unknown/empty location: PASSED')

## 8. run() — known location, no deadline

In [ ]:
response = run(location='Colombo', deadline=None)
print('run(Colombo):', response)

assert isinstance(response, str)
assert len(response) > 0
assert 'next' in response.lower() or 'day' in response.lower() or 'colombo' in response.lower(), \
    'Response should mention delivery info for Colombo'
print('run() known location: PASSED')

## 9. run() — not-covered location

In [ ]:
response = run(location='Kilinochchi', deadline=None)
print('run(Kilinochchi):', response)

assert isinstance(response, str)
assert len(response) > 0
# Should mention unavailability or no coverage
not_covered_keywords = ['not', 'unavailable', 'cover', 'unable', 'sorry']
assert any(kw in response.lower() for kw in not_covered_keywords), \
    f'Response should indicate no coverage. Got: {response}'
print('run() not-covered: PASSED')

## 10. run() — unknown location (LLM fallback)

In [ ]:
response = run(location='Atlantis', deadline=None)
print('run(Atlantis) [LLM fallback]:', response)

assert isinstance(response, str)
assert len(response) > 0
print('run() LLM fallback: PASSED')

## 11. run() — with deadline

In [ ]:
response = run(location='Kandy', deadline='tomorrow')
print('run(Kandy, tomorrow):', response)

assert isinstance(response, str)
assert len(response) > 0
print('run() with deadline: PASSED')

## 12. All 25 districts summary

In [ ]:
from collections import defaultdict

tier_groups = defaultdict(list)
not_covered_list = []

for district, info in DISTRICT_COVERAGE.items():
    if info['covered']:
        tier_groups[info['tier']].append(district)
    else:
        not_covered_list.append(district)

for tier in sorted(tier_groups):
    print(f'Tier {tier} ({TIER_DESCRIPTION[tier]}): {tier_groups[tier]}')
print(f'Not covered: {not_covered_list}')